# $c_{p,m}$ model-agreement chart — PAR wind sector (24 flat-roof buildings)

Wind-tunnel surface-pressure visualization for the RWTH Aachen / CWE
boundary-layer wind-tunnel dataset (B-reference no-parapet + C-reference
parapet building families, Brand & Kemper). **Format:** Prof. Kemper's
`PlainCharts.ipynb` prototype — stacked transparent integer layers. Each
building contributes one layer coloured by its mean pressure coefficient
class — 0 white / 1 yellow / 2 red / 3 blue on the fixed scale **[−2, +1]** —
and the overlay of all 24 layers shows model agreement: pure colour = all
buildings in the same class, mixed colour = disagreement.

## Contents

| # | Section | What it contains |
|---|---|---|
| 1 | [Imports](#1-Imports) | libraries used |
| 2 | [Environment & data location](#2-Environment--data-location) | version report, `DATA_DIR` resolution |
| 3 | [Study constants](#3-Study-constants) | 24 buildings, PAR sector angles, colour classes, grid |
| 4 | [Data access functions](#4-Data-access-functions-HDF5) | how values are read from the `.h5` files |
| 5 | [Pooling & gridding functions](#5-Sector-pooling--gridding-functions) | mirroring, interpolation, classification |
| 6 | [Load the data](#6-Load-the-data-24-buildings) | runs the pipeline for all 24 buildings |
| 7 | [Chart functions](#7-Chart-functions) | drawing the stacked-agreement panel |
| 8 | [Page 1 — the chart](#8-Page-1--the-agreement-chart) | render |
| 9 | [Page 2 — method figure](#9-Page-2--method-explanation-figure) | angles compass, tap sketch, pooled result |
| 10 | [Audit probes](#10-Audit-probe-functions) | sample locations for the value→colour audit |
| 11 | [Audit workbook (Excel)](#11-Audit-workbook-Excel) | per-angle tables with live formulas |
| 12 | [Page 3 & final PDF](#12-Page-3--assemble-the-final-PDF) | audit table page, 3-page PDF |

---

## Requirements & environment

| Component | Tested with | Notes |
|---|---|---|
| Python | **3.11.15** | any Python ≥ 3.9 should work |
| numpy | 2.4.4 | array handling |
| scipy | 1.17.1 | `griddata` interpolation, Gaussian smoothing |
| matplotlib | 3.10.9 | figures + multi-page PDF |
| h5py | 3.16.0 | reads the `*_statistical_dataset.h5` files |
| openpyxl | 3.1.5 | writes the audit Excel workbook |
| Jupyter | Lab ≥ 4 or VS Code | anything that runs `.ipynb` |

Install into a fresh environment (recommended):

```bash
python -m venv .venv
# Windows:  .venv\Scripts\activate     Linux/macOS:  source .venv/bin/activate
pip install -r requirements.txt
# or directly:
pip install numpy scipy matplotlib h5py openpyxl jupyterlab
```

`requirements.txt` (ships next to this notebook):

```text
numpy>=1.24
scipy>=1.10
matplotlib>=3.7
h5py>=3.8
openpyxl>=3.1
```

conda alternative: `conda create -n wtv python=3.11 numpy scipy matplotlib
h5py openpyxl jupyterlab`

## Data requirement (not included in this repository)

The notebook needs the **32 campaign files** `<model>_statistical_dataset.h5`
of the 24 flat-roof buildings (4 parapet groups A0/A0025/A005/A01 × 6
geometries H1L1…H2L3; the L3 geometries have `_edge` + `_center` campaigns,
one centre campaign may ship as a `.zip` — handled automatically).
≈ 2.2 GB total — **do not commit them to git** (add `*.h5` and `*.zip` to
`.gitignore`). Source: CWE / RWTH Aachen wind-tunnel dataset (DFG FLuWiK,
Brand & Kemper data paper).

`DATA_DIR` is resolved in this order:

```text
1. environment variable WTV_DATA_DIR      -> any absolute path
2. ../Dataset for Visualization           -> notebook inside e.g. WTV Final/
3. ./Dataset for Visualization            -> dataset folder next to notebook
```

```text
Wind Tunnel Visualization/
├── Dataset for Visualization/        <- the 32 .h5 files live here
│   ├── A0H1L1_statistical_dataset.h5
│   ├── ...
└── WTV Final/
    ├── WTV_cpm_agreement.ipynb       <- this notebook (run from here)
    └── requirements.txt
```

On another machine (or Google Colab after mounting Drive), set the path
before running the cells:

```python
import os
os.environ["WTV_DATA_DIR"] = r"D:\path\to\Dataset for Visualization"
```

## How to run

`jupyter lab` → open the notebook → **Run All** (≈ 2–5 min; only the 10 PAR
angles are read from each file). Outputs written next to the notebook:
`cpm_PAR_stack.pdf` (3 pages: chart / method / audit) and
`cpm_PAR_audit.xlsx` (live formulas — open once in Excel/LibreOffice so they
are computed; simple previewers may show blank cells).

## Method in one paragraph

For every building the `Cp_mean` value of each pressure tap is read for the
10 PAR wind angles (11.25° grid: 337.5, 348.75, 0, 11.25, 22.5 plus the
mirror range 157.5…202.5 pooled after mirroring about the building *y*-axis,
per the data paper Sec. 2.5 — the chart is therefore wind-relative, windward
edge at $t=-0.5$). Half-instrumented models (L2, L3) are completed by the
reflected-angle mirror. Each angle field is interpolated to a common unfolded
grid ($t = x/L$, $s = y/B + z/H$), the 10 fields are **averaged** (Kemper:
"averaged per building"; data-paper default would be the median — switch
`AGG` in Section 3), binned into the 4 classes and stacked. The statistic
column is resolved **by name** from each dataset's `Index` attribute (13-col
B-reference / 14-col C-reference layouts); `Cp_mean` is identical in the
CP-10 / CP-1 groups. Verified against the official datasheets (A0H1L1:
envelope Gumbel_Max +0.97 / Gumbel_Min −2.01 / max σ 0.287 vs sheet scales
0…1 / −2.5…0 / 0…0.3) and the θ=90° stagnation check (+0.84).


## 1. Imports

Standard scientific-Python stack only — no exotic dependencies. `h5py` reads the wind-tunnel files, `scipy` interpolates the scattered taps onto a regular grid, `openpyxl` writes the audit workbook.


In [ ]:
# imports + constants: buildings, PAR sector, colour classes, grid
import io
import os
import zipfile
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter


## 2. Environment & data location

Prints the Python/library versions (useful when reporting problems) and resolves `DATA_DIR` — see the header for the three-step resolution order. Raises a clear error if the dataset folder cannot be found.


In [ ]:
# ---- environment report (helps when sharing / debugging on other machines) --
import sys
print("Python", sys.version.split()[0],
      "| numpy", np.__version__, "| scipy", __import__("scipy").__version__,
      "| matplotlib", __import__("matplotlib").__version__,
      "| h5py", h5py.__version__)

# ---- configuration -----------------------------------------------------------
# DATA_DIR = folder containing the  <model>_statistical_dataset.h5  files.
# Resolution order (first match wins):
#   1. environment variable WTV_DATA_DIR   -> any absolute path, e.g.
#        import os; os.environ["WTV_DATA_DIR"] = r"D:\my\data"   (before Cell 1)
#   2. ../Dataset for Visualization        -> notebook lives in  WTV Final/
#   3. ./Dataset for Visualization         -> dataset folder next to notebook
_candidates = ([Path(os.environ["WTV_DATA_DIR"])] if "WTV_DATA_DIR" in os.environ
               else []) + [Path("../Dataset for Visualization"),
                           Path("Dataset for Visualization")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset folder not found. Either place the notebook so that "
        "'../Dataset for Visualization' exists (folder with the "
        "*_statistical_dataset.h5 files), or set the environment variable "
        "WTV_DATA_DIR to the folder's absolute path before running this cell.")
print("DATA_DIR ->", DATA_DIR.resolve())
OUT_DIR = Path(".")                     # outputs are written next to the notebook


## 3. Study constants

Everything that defines the study, in one place:

| Constant | Meaning |
|---|---|
| `PARAPETS`, `GEOMS`, `BUILDINGS` | the 24 buildings: 4 parapet groups × 6 geometries |
| `PRIMARY`, `MIRROR` | the 10 PAR angle indices (±22.5° about 0°/180°, data paper Sec. 2.5) |
| `AGG` | sector aggregate per building — `np.nanmean` (Kemper), switch to `np.nanmedian` for the paper default |
| `BIN_EDGES`, `COLORS_RGB` | the four cp,m classes on the fixed scale −2…+1 |
| `TI`, `SI` | the common unfolded grid: t = x/L, s = y/B + z/H |


In [ ]:
# ---- building inventory: 4 parapet groups x 6 geometries = 24 buildings -----
PARAPETS = [("A0", "no parapet"), ("A0025", "hp/h=0.025"),
            ("A005", "hp/h=0.05"), ("A01", "hp/h=0.10")]
GEOMS = [("H1L1", "L1"), ("H1L2", "L2"), ("H1L3", "L3"),
         ("H2L1", "L1"), ("H2L2", "L2"), ("H2L3", "L3")]
BUILDINGS = [(p, g, k) for p, _ in PARAPETS for g, k in GEOMS]

# ---- PAR sector: angle-group indices into the 32 sorted angles --------------
PRIMARY = [30, 31, 0, 1, 2]      # 337.50 348.75   0.00  11.25  22.50 deg
MIRROR  = [14, 15, 16, 17, 18]   # 157.50 168.75 180.00 191.25 202.50 deg
ANGLE_LABELS = ["337.5", "348.75", "0", "11.25", "22.5",
                "157.5*", "168.75*", "180*", "191.25*", "202.5*"]  # * = mirrored
AGG = np.nanmean

# ---- cp,m classes (Prof. Kemper's scale: -2 bottom/white .. +1 top/blue) ----
BIN_EDGES  = np.array([-2.0, -1.25, -0.5, 0.25, 1.0])
COLORS_RGB = [(1.0, 1.0, 1.0),   # 0 white  : strongest mean suction
              (1.0, 1.0, 0.0),   # 1 yellow
              (1.0, 0.0, 0.0),   # 2 red
              (0.0, 0.0, 1.0)]   # 3 blue   : mean overpressure
COLOR_NAMES = ["white", "yellow", "red", "blue"]

# ---- common unfolded grid ----------------------------------------------------
TI = np.linspace(-0.5, 0.5, 120)     # t = x/L
SI = np.linspace(-1.5, 1.5, 360)     # s = y/B + z/H (unwinded)
TG, SG = np.meshgrid(TI, SI)


## 4. Data access functions (HDF5)

How a number gets out of the database:

| Function | What it does | Returns |
|---|---|---|
| `open_h5(path)` | opens an `.h5` file, or the `.h5` inside a `.zip` | `h5py.File` |
| `side_s(name, gy, gz, Wy, H)` | tap side letter (N=roof / P=Wall+ / M=Wall−) → unfolded coordinate s | `float` |
| `read_cpm(path)` | reads the `Cp_mean` column (resolved **by name** from the `Index` attribute, so 13-col B-ref and 14-col C-ref both work) for the 10 PAR angles, every tap | `t, s, cp[32, n_tap]` |
| `sources(prefix, geom, kind)` | which campaign file(s) belong to a building (L3 = edge + centre) | list of paths |


In [ ]:
# ---- HDF reading -------------------------------------------------------------
def open_h5(path):
    """Open an .h5 file, or the single .h5 member inside a .zip."""
    path = str(path)
    if path.endswith(".zip"):
        z = zipfile.ZipFile(path)
        member = [n for n in z.namelist() if n.endswith(".h5")][0]
        return h5py.File(io.BytesIO(z.read(member)), "r")
    return h5py.File(path, "r", locking=False)


def side_s(name, gy, gz, Wy, H):
    """Unfolded perimeter coordinate s from the tap side letter (N/P/M)."""
    sd = name.split("-")[1]
    if sd == "N":
        return gy / Wy               # roof  : s in [-0.5, 0.5]
    if sd == "P":
        return 1.5 - gz / H          # Wall+ : s in [ 0.5, 1.5]
    if sd == "M":
        return -1.5 + gz / H         # Wall- : s in [-1.5,-0.5]
    raise ValueError(name)


NEEDED_ANGLES = sorted(set(PRIMARY) | set(MIRROR))   # only these 10 are read


def read_cpm(path):
    """One campaign file -> (t, s, cp[32, n_tap]) of `Cp_mean` per angle.
    Only the NEEDED_ANGLES rows are filled (3x faster than reading all 32);
    the array keeps its (32, n_tap) shape so angle indices stay absolute."""
    f = open_h5(path)
    O = f["Info/Geometry/Original"][()]
    Lx = O["GlobalX"].max() - O["GlobalX"].min()
    Wy = O["GlobalY"].max() - O["GlobalY"].min()
    H  = O["GlobalZ"].max() - O["GlobalZ"].min()
    pos = {}
    for row in O:
        if row["IsMeasurementPoint"] == 1:
            nm = row["PointID"]
            nm = nm.decode() if isinstance(nm, bytes) else nm
            pos[nm] = (float(row["GlobalX"]), float(row["GlobalY"]),
                       float(row["GlobalZ"]))
    angles = sorted(k for k in f.keys() if k != "Info")
    names = sorted(n for n in f[angles[0]]["CP-10"].keys() if n in pos)
    ci = None
    cp = np.full((32, len(names)), np.nan)
    for ai in NEEDED_ANGLES:
        g = f[angles[ai]]["CP-10"]               # Cp_mean identical in CP-1
        for ni, n in enumerate(names):
            d = g[n]
            if ci is None:
                idx = [x.decode() if isinstance(x, bytes) else x
                       for x in d.attrs["Index"]]
                ci = idx.index("Cp_mean")
            cp[ai, ni] = float(d[()][ci])
    f.close()
    t = np.array([pos[n][0] / Lx for n in names])
    s = np.array([side_s(n, pos[n][1], pos[n][2], Wy, H) for n in names])
    return t, s, cp


def sources(prefix, geom, kind):
    """Campaign file(s) of one building (L3 = edge + centre campaigns)."""
    base = prefix + geom
    if kind in ("L1", "L2"):
        return [DATA_DIR / f"{base}_statistical_dataset.h5"]
    out = [DATA_DIR / f"{base}_edge_statistical_dataset.h5"]
    for cand in (DATA_DIR / f"{base}_center_statistical_dataset.h5",
                 DATA_DIR / f"{base}_center_dataset.zip"):
        if cand.exists():
            out.append(cand)
            break
    return out


## 5. Sector pooling & gridding functions

From per-tap values to a per-building field:

| Function | What it does | Returns |
|---|---|---|
| `refl(a)` | reflected angle index (θ → 180°−θ), used to complete half-instrumented models | `int` |
| `angle_points(t, s, cp, a, complete)` | scattered points of one angle; adds the mirror-reconstructed half for L2/L3 | `T, S, V` |
| `grid_one(T, S, V)` | scattered taps → regular grid (linear + nearest fill, σ=3 px smoothing) | 2-D array |
| `building_angle_fields(prefix, geom, kind)` | the 10 gridded PAR fields; the 157.5…202.5° family is mirrored about y (t→−t) per the data paper | `(10, ns, nt)` |
| `to_layer(Z)` | cp,m field → integer classes 0…3 (values clipped into −2…+1) | int array |


In [ ]:
# ---- PAR-sector pooling ------------------------------------------------------
def refl(a):
    """Angle index reflected through the mid-length plane (theta -> 180-theta)."""
    return (16 - a) % 32


def angle_points(t, s, cp, a, complete):
    """Scattered points of angle a; `complete` mirror-reconstructs the
    un-instrumented half (L2 + L3 campaigns) from angle refl(a)."""
    T, S, V = [t], [s], [cp[a]]
    if complete:
        T.append(-t); S.append(s); V.append(cp[refl(a)])
    return np.concatenate(T), np.concatenate(S), np.concatenate(V)


def grid_one(T, S, V, smooth=3.0):
    """Scattered taps -> regular (t, s) grid (linear + nearest fill,
    light Gaussian smoothing -- identical to the map pipeline)."""
    Z = griddata((T, S), V, (TG, SG), method="linear")
    Zn = griddata((T, S), V, (TG, SG), method="nearest")
    Z = np.where(np.isnan(Z), Zn, Z)
    return gaussian_filter(Z, sigma=smooth, mode="nearest") if smooth else Z


def building_angle_fields(prefix, geom, kind):
    """The 10 gridded PAR angle fields of one building, mirror range already
    mirrored about y (t -> -t). Returns array (10, ns, nt)."""
    camps = [read_cpm(p) for p in sources(prefix, geom, kind)]
    complete = kind != "L1"                  # L2/L3 are half-instrumented
    fields = []
    for group, sign in ((PRIMARY, +1), (MIRROR, -1)):
        for a in group:
            Ts, Ss, Vs = [], [], []
            for (t, s, cp) in camps:
                T, S, V = angle_points(t, s, cp, a, complete)
                Ts.append(T); Ss.append(S); Vs.append(V)
            fields.append(grid_one(sign * np.concatenate(Ts),
                                   np.concatenate(Ss), np.concatenate(Vs)))
    assert len(fields) == 10, "PAR sector must pool exactly 10 angles"
    return np.stack(fields)


def to_layer(Z):
    """cp,m field -> integer classes 0..3 (clipped into [-2, +1])."""
    Zc = np.clip(Z, BIN_EDGES[0] + 1e-9, BIN_EDGES[-1] - 1e-9)
    return np.digitize(Zc, BIN_EDGES[1:-1]).astype(np.uint8)


## 6. Load the data (24 buildings)

Checks that all 32 campaign files exist, then runs the pipeline for every building and prints each building's sector-mean cp,m range as a quick sanity check (PAR means stay within ≈ −1.35…+0.35). Results are kept in `STACKS` (per-angle fields, used by the audit), `MEANS` and `LAYERS` (the class maps that get stacked).


In [ ]:
# ---- build all 24 buildings (angle stacks kept for the audit cell) ----------
# early check first: are all 32 campaign files reachable?
_missing = [Path(p).name for prefix, geom, kind in BUILDINGS
            for p in sources(prefix, geom, kind) if not Path(p).exists()]
if _missing:
    raise FileNotFoundError(
        f"{len(_missing)} dataset file(s) missing in '{DATA_DIR}':\n  "
        + "\n  ".join(_missing))
print("all campaign files found")

STACKS, MEANS, LAYERS, LABELS = {}, {}, {}, []
for prefix, geom, kind in BUILDINGS:
    name = prefix + geom
    stack = building_angle_fields(prefix, geom, kind)
    STACKS[name] = stack
    MEANS[name] = AGG(stack, axis=0)
    LAYERS[name] = to_layer(MEANS[name])
    LABELS.append(name)
    print(f"  {name:12s} cp,m (PAR mean) range: "
          f"{MEANS[name].min():+.2f} .. {MEANS[name].max():+.2f}", flush=True)
print(f"{len(LABELS)} buildings ready")


## 7. Chart functions

| Function | What it does |
|---|---|
| `layer_rgba(layer, alpha)` | one building's class map → semi-transparent RGBA image |
| `plot_agreement_panel(layers, …)` | stacks all layers, draws the wall/roof separators, u∞ arrow, single cp,m colourbar and caption |


In [ ]:
# plotting helpers for the stacked-agreement chart
FONTSIZE = 10


def layer_rgba(layer, alpha):
    pal = np.asarray(COLORS_RGB, dtype=float)
    rgba = np.zeros((*layer.shape, 4))
    rgba[..., :3] = pal[layer]
    rgba[..., 3] = alpha
    return rgba


def plot_agreement_panel(layers, title="PAR", fontsize=FONTSIZE,
                         fig_width_cm=9.0):
    n = len(layers)
    alpha = 2.2 / n                       # readable for a 24-layer stack
    plt.rcParams.update({"font.size": fontsize})
    fig_w = fig_width_cm / 2.54
    fig, ax = plt.subplots(figsize=(fig_w, fig_w * 1.32))
    ax.set_facecolor("white")
    for L in layers:                      # one imshow per building
        ax.imshow(layer_rgba(L, alpha), origin="lower",
                  extent=[TI.min(), TI.max(), SI.min(), SI.max()],
                  interpolation="nearest", aspect="auto", zorder=2)
    ax.axhline(-0.5, color="w", lw=1.2, zorder=5)     # wall/roof separators
    ax.axhline(0.5, color="w", lw=1.2, zorder=5)
    ax.set_title(title, pad=0.5 * fontsize)
    ax.set_xlim(-0.5, 0.5); ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel(r"$t = x/L$"); ax.set_xticks([-0.5, 0.0, 0.5])
    ax.set_ylabel(r"$s = y/B + z/H$ (unwinded)")
    ax.set_yticks([-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5])
    ax.set_box_aspect(1.4)

    # wind direction: PAR pooling orients every angle so the flow comes from
    # the -t side (windward gable edge at t = -0.5); arrow below-left of panel
    ax.annotate("", xy=(-0.38, -1.88), xytext=(-0.50, -1.88),
                xycoords="data", annotation_clip=False,
                arrowprops=dict(arrowstyle="-|>", color="0.25", lw=1.4))
    ax.text(-0.345, -1.88, r"$u_\infty$", ha="left", va="center",
            fontsize=fontsize - 1, color="0.25", clip_on=False)

    # single colourbar in cp,m units (no separate 0-3 axis)
    cmap = ListedColormap(COLORS_RGB)
    norm = BoundaryNorm(BIN_EDGES, cmap.N)
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.055, pad=0.18)
    cbar.set_ticks(BIN_EDGES)
    cbar.set_ticklabels([f"{e:+.2f}" for e in BIN_EDGES])
    cbar.ax.tick_params(labelsize=fontsize - 1)
    cbar.set_label(r"$c_{p,m}$  class  [-]", fontsize=fontsize)
    cbar.outline.set_linewidth(0.8)

    # region labels
    bbox = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85)
    for sc, txt in [(1.0, "Wall+"), (0.0, "Roof"), (-1.0, "Wall-")]:
        ax.text(0.56, sc, txt, rotation=90, va="center", ha="center",
                fontsize=fontsize - 1, bbox=bbox, clip_on=False)

    # caption: how to read the chart
    fig.text(0.5, 0.03,
             "each of the %d buildings adds one transparent layer coloured by its "
             "$c_{p,m}$ class;\npure colour = all buildings in the same class, "
             "mixed colour = disagreement between buildings" % n,
             ha="center", va="top", fontsize=fontsize - 2, color="0.35")
    fig.subplots_adjust(bottom=0.125, top=0.93)
    return fig


## 8. Page 1 — the agreement chart

Renders the PAR panel with all 24 buildings and saves a PNG preview.


In [ ]:
# render page 1
fig1 = plot_agreement_panel([LAYERS[m] for m in LABELS],
                            title=f"PAR   ($c_{{p,m}}$, {len(LABELS)} buildings)")
fig1.savefig(OUT_DIR / "cpm_PAR_stack_page1.png", dpi=170, bbox_inches="tight")
plt.show()


## 9. Page 2 — method explanation figure

One self-contained figure that answers the three questions reviewers ask: **which** 10 of the 32 measured angles enter the PAR sector (compass), **why** no positive pressure appears (only Wall+/roof/Wall− carry taps; the gable ends facing parallel wind are not instrumented), and **why** the chart is one-sided (the 180° family is mirrored onto the 0° orientation per the data paper — wind-relative frame, windward edge left).


In [ ]:
# the method-explanation figure (page 2)
from matplotlib.patches import Wedge, Rectangle, FancyArrow

ALL_DEG = [11.25 * k for k in range(32)]
PAR_DEG_PRIM = [337.5, 348.75, 0.0, 11.25, 22.5]
PAR_DEG_MIRR = [157.5, 168.75, 180.0, 191.25, 202.5]
NAMED = {337.5, 0.0, 22.5, 157.5, 180.0, 202.5}


def _disp(theta_deg):
    """Display angle of the 'wind FROM theta' position on the compass:
    theta=0 comes from -x (left), theta=90 from +y (top)  ->  180 - theta."""
    return np.deg2rad(180.0 - theta_deg)


def plot_method_page(fontsize=9):
    fig = plt.figure(figsize=(8.27, 10.6))

    # ------------------------------------------------ compass + plan sketch --
    ax = fig.add_axes([0.08, 0.42, 0.84, 0.52])
    ax.set_aspect("equal"); ax.set_axis_off()
    ax.set_xlim(-2.35, 2.35); ax.set_ylim(-1.95, 1.95)
    R = 1.52

    # sector bands (display angles: primary band = left arc, mirror = right)
    ax.add_patch(Wedge((0, 0), R + 0.13, 157.5, 202.5, width=0.26,
                       fc="#b7e4b0", ec="none", alpha=0.9))
    ax.add_patch(Wedge((0, 0), R + 0.13, -22.5, 22.5, width=0.26,
                       fc="#ffd9a0", ec="none", alpha=0.9))

    for th in ALL_DEG:
        x, y = R * np.cos(_disp(th)), R * np.sin(_disp(th))
        used = th in PAR_DEG_PRIM or th in PAR_DEG_MIRR
        col = ("#2e7d32" if th in PAR_DEG_PRIM else
               "#e07b00" if th in PAR_DEG_MIRR else "0.75")
        ax.plot(x, y, "o", ms=7 if used else 4.5, mfc=col if used else "white",
                mec=col, mew=1.4, zorder=5)
        if used:
            lab = f"{th:g}".replace(".0", "") if th == int(th) else f"{th:g}"
            xx, yy = (R + 0.30) * np.cos(_disp(th)), (R + 0.30) * np.sin(_disp(th))
            ax.text(xx, yy, lab + "°", ha="center", va="center",
                    fontsize=fontsize - 1 if th in NAMED else fontsize - 2.5,
                    fontweight="bold" if th in NAMED else "normal",
                    color="#1b5e20" if th in PAR_DEG_PRIM else "#a35a00")

    # building plan: instrumented Wall+ / roof / Wall-, bare gable ends
    L, B = 1.30, 0.86
    ax.add_patch(Rectangle((-L/2, -B/2), L, B, fc="#f7edc8", ec="0.2", lw=1.2))
    ax.add_patch(Rectangle((-L/2, B/2 - 0.13), L, 0.13, fc="#7c86e8",
                           ec="0.2", lw=0.8))
    ax.add_patch(Rectangle((-L/2, -B/2), L, 0.13, fc="#e86a6a", ec="0.2",
                           lw=0.8))
    for xg in (-L/2, L/2):
        ax.plot([xg, xg], [-B/2, B/2], color="0.35", lw=5, solid_capstyle="butt")
    ax.text(0, B/2 - 0.065, "Wall+ (taps)", ha="center", va="center",
            fontsize=fontsize - 1.5)
    ax.text(0, -B/2 + 0.065, "Wall- (taps)", ha="center", va="center",
            fontsize=fontsize - 1.5)
    ax.text(0, 0.05, "roof (taps)", ha="center", va="center",
            fontsize=fontsize - 1)
    ax.text(0, -0.16, "gable ends: NO taps", ha="center", va="center",
            fontsize=fontsize - 1.5, color="0.35", style="italic")
    ax.annotate("", xy=(-L/2 - 0.02, -0.32), xytext=(-L/2 - 0.30, -0.55),
                arrowprops=dict(arrowstyle="-", color="0.35", lw=0.8))
    ax.text(-L/2 - 0.32, -0.62, "stagnation (+) lives here,\nbut is not measured",
            ha="center", va="top", fontsize=fontsize - 2, color="0.35")

    # wind arrows of the two families
    ax.add_patch(FancyArrow(-R + 0.28, 0, 0.42, 0, width=0.015,
                            head_width=0.09, head_length=0.10,
                            fc="#2e7d32", ec="#2e7d32"))
    ax.text(-R + 0.46, 0.14, "0°-family\n(5 angles)", ha="center",
            fontsize=fontsize - 1.5, color="#1b5e20")
    ax.add_patch(FancyArrow(R - 0.28, 0.22, -0.42, 0, width=0.015,
                            head_width=0.09, head_length=0.10,
                            fc="#e07b00", ec="#e07b00"))
    ax.text(R - 0.48, 0.36, "180°-family\n(5 angles)", ha="center",
            fontsize=fontsize - 1.5, color="#a35a00")
    ax.annotate("mirrored about the y-axis (t → −t)\nbefore pooling "
                "(table: 'Mirror axis y')",
                xy=(R - 0.62, 0.16), xytext=(0.72, -1.18),
                fontsize=fontsize - 1.5, color="#a35a00", ha="center",
                arrowprops=dict(arrowstyle="->", color="#a35a00", lw=1.0,
                                connectionstyle="arc3,rad=0.3"))
    ax.plot([0, 0], [-B/2 - 0.22, B/2 + 0.22], color="0.5", lw=0.8, ls=":")
    ax.text(0.05, B/2 + 0.26, "y", fontsize=fontsize - 1, color="0.4")

    ax.set_title("PAR sector on the 11.25° measurement grid — "
                 "10 of 32 angles are pooled (±22.5° = ±2 steps about 0°/180°)",
                 fontsize=fontsize + 1, pad=8)

    # ------------------------- the pooled result in the mirrored orientation --
    axm = fig.add_axes([0.36, 0.075, 0.28, 0.30])
    alpha = 2.2 / len(LABELS)
    for name in LABELS:
        axm.imshow(layer_rgba(LAYERS[name], alpha), origin="lower",
                   extent=[TI.min(), TI.max(), SI.min(), SI.max()],
                   interpolation="nearest", aspect="auto", zorder=2)
    axm.axhline(-0.5, color="w", lw=1.0); axm.axhline(0.5, color="w", lw=1.0)
    axm.set_xlim(-0.5, 0.5); axm.set_ylim(-1.5, 1.5)
    axm.set_box_aspect(1.25)
    axm.set_xticks([-0.5, 0, 0.5]); axm.set_yticks([-1.5, 0, 1.5])
    axm.tick_params(labelsize=fontsize - 2)
    axm.set_xlabel(r"$t=x/L$", fontsize=fontsize - 1)
    axm.set_title("pooled PAR result (mirrored orientation)",
                  fontsize=fontsize, pad=4)
    axm.annotate("", xy=(-0.30, -1.82), xytext=(-0.50, -1.82),
                 xycoords="data", annotation_clip=False,
                 arrowprops=dict(arrowstyle="-|>", color="0.25", lw=1.3))
    axm.text(-0.26, -1.82, r"$u_\infty$", ha="left", va="center",
             fontsize=fontsize - 1, color="0.25", clip_on=False)
    return fig


fig_expl = plot_method_page()
fig_expl.savefig(OUT_DIR / "cpm_PAR_stack_page2_method.png", dpi=170,
                 bbox_inches="tight")
plt.show()


## 10. Audit probe functions

Six probe locations (windward/centre/leeward roof, both walls) used to document how values become colours.

| Function | What it does | Returns |
|---|---|---|
| `probe_indices(t, s)` | nearest grid node of a probe | `(js, jt)` |
| `probe_values(name, t, s)` | one building's 10 per-angle cp,m values at a probe (sampled from the same gridded fields as the chart) | `(10,)` array |


In [ ]:
# audit probe locations + sampling helpers
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PROBES = [  # (short, description, t, s)  -- P3 sits inside the leeward zone
    ("P1", "Roof windward edge",  -0.45,  0.00),
    ("P2", "Roof centre",          0.00,  0.00),
    ("P3", "Roof leeward edge",   +0.48,  0.00),
    ("P4", "Wall+ mid-height",     0.00, +1.00),
    ("P5", "Wall- mid-height",     0.00, -1.00),
    ("P6", "Wall+ windward edge", -0.45, +1.00),
]
SHORT_DESC = {"P1": "Roof windward", "P2": "Roof centre", "P3": "Roof leeward",
              "P4": "Wall+ mid", "P5": "Wall- mid", "P6": "Wall+ windward"}

HEX = ["FFFFFF", "FFFF00", "FF0000", "0000FF"]          # class fills
TXT = ["000000", "000000", "FFFFFF", "FFFFFF"]          # readable text on fill


def probe_indices(t, s):
    """Nearest grid node of the common (TI, SI) grid."""
    return int(np.argmin(np.abs(SI - s))), int(np.argmin(np.abs(TI - t)))


def probe_values(name, t, s):
    """The 10 per-angle cp,m values of one building at one probe."""
    js, jt = probe_indices(t, s)
    return STACKS[name][:, js, jt]


## 11. Audit workbook (Excel)

Writes `cpm_PAR_audit.xlsx`: a README sheet (method + class table), one sheet per probe (24 buildings × 10 angle values, with Mean / Class / Colour as **live formulas**), and a Summary sheet with COUNTIF class counts and the resulting colour mix. Open once in Excel/LibreOffice to compute the formulas.


In [ ]:
# ---------------------------------------------------------------- Excel ----
ARIAL = "Arial"
thin = Side(style="thin", color="BBBBBB")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)


def style(cell, bold=False, fill=None, color="000000", fmt=None,
          center=False, border=True):
    cell.font = Font(name=ARIAL, size=10, bold=bold, color=color)
    if fill:
        cell.fill = PatternFill("solid", fgColor=fill)
    if fmt:
        cell.number_format = fmt
    if center:
        cell.alignment = Alignment(horizontal="center", vertical="center")
    if border:
        cell.border = BORDER


wb = Workbook()

# ---- README sheet -----------------------------------------------------------
ws = wb.active
ws.title = "README"
readme = [
    ("cp,m model-agreement chart - PAR sector - audit workbook", True),
    ("", False),
    ("Pipeline (identical to the chart):", True),
    ("1. Source: <model>_statistical_dataset.h5, statistic column 'Cp_mean' "
     "resolved by name from each dataset's Index attribute (group CP-10; "
     "Cp_mean is identical in CP-1 because a mean is unaffected by the TVL "
     "area-averaging).", False),
    ("2. PAR sector = 10 wind angles (11.25 deg grid): 337.5, 348.75, 0, "
     "11.25, 22.5 plus the mirror range 157.5...202.5 (data paper Sec. 2.5); "
     "mirror-range fields are mirrored about the building y-axis (t -> -t) "
     "before pooling. Angles marked * in the sheets are the mirrored ones.", False),
    ("3. Half-instrumented models (L2, L3 campaigns): the missing half is "
     "completed with the value of the reflected angle refl(a) = 180 - a "
     "placed at (-t, s).", False),
    ("4. Each angle field is interpolated to a regular grid (linear + "
     "nearest fill) and smoothed (Gaussian, sigma = 3 px) - probe values "
     "here are sampled from exactly those fields, so they reproduce the "
     "chart, not the raw tap readings.", False),
    ("5. Sector aggregate per building = MEAN of the 10 angle values "
     "(column 'Mean cp,m' = AVERAGE formula).", False),
    ("6. Class = IF(mean < -1.25, 0, IF(mean < -0.5, 1, IF(mean < 0.25, 2, "
     "3))); values below -2 / above +1 clip into the end classes.", False),
    ("7. Colour: 0 = white, 1 = yellow, 2 = red, 3 = blue. In the chart every "
     "building adds one transparent layer of its colour; the class counts in "
     "'Summary' explain the mixed colour at each probe.", False),
    ("", False),
    ("Class boundaries (Prof. Kemper's scale):", True),
    ("   class 0  white  :  -2.00 <= cp,m < -1.25   (strongest mean suction)", False),
    ("   class 1  yellow :  -1.25 <= cp,m < -0.50", False),
    ("   class 2  red    :  -0.50 <= cp,m < +0.25", False),
    ("   class 3  blue   :  +0.25 <= cp,m <= +1.00  (mean overpressure)", False),
    ("", False),
    ("Note: the official datasheets show ENVELOPES of the characteristic "
     "extremes (max cp_hat,10 / min cp_check,10 / max sigma over all 32 "
     "directions) - those are peak statistics and therefore much larger in "
     "magnitude than the mean cp,m shown here (A0H1L1 verification: envelope "
     "min Gumbel_Min = -2.01 vs datasheet scale -2.5..0; envelope max "
     "Gumbel_Max = +0.97 vs scale 0..1; max sigma = 0.287 vs scale 0..0.3).", False),
]
for r, (txt, bold) in enumerate(readme, start=1):
    c = ws.cell(row=r, column=1, value=txt)
    style(c, bold=bold, border=False)
ws.column_dimensions["A"].width = 130
for r in range(1, len(readme) + 1):
    ws.row_dimensions[r].height = 28 if len(readme[r - 1][0]) > 90 else 14
    ws.cell(row=r, column=1).alignment = Alignment(wrap_text=True,
                                                   vertical="top")

# ---- one sheet per probe ------------------------------------------------------
probe_sheetnames = {}
for short, desc, tt, ss in PROBES:
    sname = f"{short} {desc}"[:31].replace("+", "plus").replace("-", "minus") \
        if False else f"{short} {desc}"[:31]
    probe_sheetnames[short] = sname
    ws = wb.create_sheet(sname)
    heads = (["Model"] + [a + " deg" for a in ANGLE_LABELS]
             + ["Mean cp,m", "Class 0-3", "Colour"])
    for c, h in enumerate(heads, start=1):
        cell = ws.cell(row=1, column=c, value=h)
        style(cell, bold=True, fill="DDDDDD", center=True)
    for r, name in enumerate(LABELS, start=2):
        vals = probe_values(name, tt, ss)
        style(ws.cell(row=r, column=1, value=name), bold=True)
        for c, v in enumerate(vals, start=2):
            # full precision stored (a rounded value could flip a class at a
            # bin boundary); the number format displays 3 decimals
            style(ws.cell(row=r, column=c, value=float(v)),
                  fmt="+0.000;-0.000;0.000", center=True)
        mcell = ws.cell(row=r, column=12, value=f"=AVERAGE(B{r}:K{r})")
        style(mcell, fmt="+0.000;-0.000;0.000", center=True, bold=True)
        ccell = ws.cell(
            row=r, column=13,
            value=f"=IF(L{r}<-1.25,0,IF(L{r}<-0.5,1,IF(L{r}<0.25,2,3)))")
        style(ccell, center=True)
        ncell = ws.cell(row=r, column=14,
                        value=f'=CHOOSE(M{r}+1,"white","yellow","red","blue")')
        k = int(to_layer(np.array([[vals.mean()]]))[0, 0])   # fill = class colour
        style(ncell, fill=HEX[k], color=TXT[k], center=True, bold=True)
    note = (f"probe at t = {tt:+.2f}, s = {ss:+.2f} ({desc}); per-angle values "
            "sampled from the gridded, smoothed fields used by the chart "
            "(see README). * = angle pooled after mirroring about y (t -> -t).")
    c = ws.cell(row=len(LABELS) + 3, column=1, value=note)
    style(c, border=False, color="666666")
    ws.column_dimensions["A"].width = 13
    for c in range(2, 15):
        ws.column_dimensions[get_column_letter(c)].width = 11

# ---- summary sheet -------------------------------------------------------------
ws = wb.create_sheet("Summary")
wb.move_sheet("Summary", offset=-(len(wb.sheetnames) - 2))   # after README
heads = ["Probe", "Location (t, s)", "n white", "n yellow", "n red", "n blue",
         "Mixed colour (mean RGB of 24 layers)", "Reading"]
for c, h in enumerate(heads, start=1):
    style(ws.cell(row=1, column=c, value=h), bold=True, fill="DDDDDD",
          center=True)
for r, (short, desc, tt, ss) in enumerate(PROBES, start=2):
    sname = probe_sheetnames[short]
    style(ws.cell(row=r, column=1, value=f"{short} {desc}"), bold=True)
    style(ws.cell(row=r, column=2, value=f"({tt:+.2f}, {ss:+.2f})"),
          center=True)
    for k in range(4):
        f = f"=COUNTIF('{sname}'!$M$2:$M$25,{k})"
        style(ws.cell(row=r, column=3 + k, value=f), center=True)
    counts = np.bincount(
        [int(to_layer(np.array([[probe_values(m, tt, ss).mean()]]))[0, 0])
         for m in LABELS], minlength=4)
    rgb = (np.array(COLORS_RGB, float).T @ counts / counts.sum())
    hexmix = "".join(f"{int(round(255 * v)):02X}" for v in rgb)
    sw = ws.cell(row=r, column=7, value="#" + hexmix)
    style(sw, fill=hexmix, center=True,
          color="FFFFFF" if rgb.mean() < 0.5 else "000000")
    parts = [f"{n} x {COLOR_NAMES[k]}" for k, n in enumerate(counts) if n]
    style(ws.cell(row=r, column=8, value=" + ".join(parts)), border=False)
ws.column_dimensions["A"].width = 24
ws.column_dimensions["B"].width = 15
for c in range(3, 7):
    ws.column_dimensions[get_column_letter(c)].width = 9
ws.column_dimensions["G"].width = 32
ws.column_dimensions["H"].width = 42
note = ("counts are live COUNTIF formulas over the probe sheets; the mixed "
        "colour approximates the stacked appearance at that spot in the chart")
style(ws.cell(row=len(PROBES) + 3, column=1, value=note), border=False,
      color="666666")

xlsx_path = OUT_DIR / "cpm_PAR_audit.xlsx"
wb.save(xlsx_path)
print("wrote", xlsx_path)


## 12. Page 3 & assemble the final PDF

Renders the compact audit table (24 buildings × 6 probes, cell colour = class, bottom rows = class counts) and writes the final 3-page `cpm_PAR_stack.pdf`: chart · method · audit.


In [ ]:
# ------------------------------------------------------------- PDF page 2 ----
def plot_audit_page(fontsize=8.5):
    """Compact rendering of the audit: 24 models x 6 probes, cell = sector-mean
    cp,m coloured by its class; bottom = class counts = the colour mix."""
    fig, ax = plt.subplots(figsize=(8.27, 10.5))
    ax.set_axis_off()
    nrow, ncol = len(LABELS), len(PROBES)
    x0, y0, w, h = 0.175, 0.86, 0.128, 0.026
    ax.text(0.5, 0.985, "How the colours are made -- audit at 6 probe locations",
            transform=fig.transFigure, ha="center", va="top", fontsize=12,
            fontweight="bold")
    ax.text(0.5, 0.955,
            "cell value = mean $c_{p,m}$ over the 10 PAR angles; cell colour = "
            "its class; bottom rows = class counts over the 24 buildings\n"
            "(= the colour mix visible at that spot of the chart). Full "
            "per-angle detail: cpm_PAR_audit.xlsx",
            transform=fig.transFigure, ha="center", va="top",
            fontsize=fontsize - 0.5, color="0.35")
    for j, (short, desc, tt, ss) in enumerate(PROBES):
        ax.text(x0 + (j + 0.5) * w, y0 + 0.008,
                f"{short}  {SHORT_DESC[short]}\n(t={tt:+.2f}, s={ss:+.2f})",
                transform=fig.transFigure, ha="center", va="bottom",
                fontsize=fontsize - 1.6, fontweight="bold")
    for i, name in enumerate(LABELS):
        y = y0 - i * h
        if i % 6 == 0:                                     # parapet group label
            ax.text(0.002, y - 0.5 * h, PARAPETS[i // 6][1],
                    transform=fig.transFigure, ha="left", va="center",
                    fontsize=fontsize - 2, fontweight="bold", color="#234")
        ax.text(x0 - 0.012, y - 0.5 * h, name, transform=fig.transFigure,
                ha="right", va="center", fontsize=fontsize - 1)
        for j, (short, desc, tt, ss) in enumerate(PROBES):
            m = float(probe_values(name, tt, ss).mean())
            k = int(to_layer(np.array([[m]]))[0, 0])
            ax.add_patch(plt.Rectangle(
                (x0 + j * w, y - h), w, h, transform=fig.transFigure,
                facecolor=COLORS_RGB[k], alpha=0.55, edgecolor="0.75",
                lw=0.4, clip_on=False))
            ax.text(x0 + (j + 0.5) * w, y - 0.5 * h, f"{m:+.2f}",
                    transform=fig.transFigure, ha="center", va="center",
                    fontsize=fontsize - 1)
    yb = y0 - nrow * h - 0.012
    for k in range(4):
        y = yb - k * 0.018
        ax.text(x0 - 0.012, y, f"n {COLOR_NAMES[k]}",
                transform=fig.transFigure, ha="right", va="center",
                fontsize=fontsize - 1, fontweight="bold")
        for j, (short, desc, tt, ss) in enumerate(PROBES):
            cnt = sum(int(to_layer(np.array([[float(
                probe_values(m_, tt, ss).mean())]]))[0, 0]) == k
                for m_ in LABELS)
            ax.text(x0 + (j + 0.5) * w, y, str(cnt),
                    transform=fig.transFigure, ha="center", va="center",
                    fontsize=fontsize - 1,
                    color="0.2" if cnt else "0.65")
    return fig


fig2 = plot_audit_page()
fig2.savefig(OUT_DIR / "cpm_PAR_stack_page2.png", dpi=170, bbox_inches="tight")

# ---- final 3-page PDF ---------------------------------------------------------
#  page 1 chart -- page 2 method explanation -- page 3 audit table
pdf_path = OUT_DIR / "cpm_PAR_stack.pdf"
with PdfPages(pdf_path) as pdf:
    pdf.savefig(fig1, bbox_inches="tight")
    pdf.savefig(fig_expl, bbox_inches="tight")
    pdf.savefig(fig2, bbox_inches="tight")
print("wrote", pdf_path, "(3 pages)")
plt.show()


## Next steps
- **PERP and DIAG columns** to complete the 3-panel page (sector index sets
  follow the same data-paper table; the mirror transforms need one final
  cross-check of the axis notation before delivery).
- Then the remaining statistics, one page each: $\sigma_{c_pm}$ , skewness, kurtosis — all reuse this pipeline.
